In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Load data
clean_df = pd.read_csv('clean_data.csv')

# Check for NaN values in text_akhir
print(f"Initial data shape: {clean_df.shape}")
print(f"NaN values in 'text_akhir': {clean_df['text_akhir'].isna().sum()}")
print(f"NaN values in 'polarity': {clean_df['polarity'].isna().sum()}")

# Remove rows with NaN in text_akhir or polarity
clean_df = clean_df.dropna(subset=['text_akhir', 'polarity'])
print(f"Shape after removing NaN: {clean_df.shape}")

# Also remove empty strings or whitespace-only strings
clean_df = clean_df[clean_df['text_akhir'].str.strip().astype(bool)]
print(f"Shape after removing empty strings: {clean_df.shape}")

# Prepare features and labels
X = clean_df['text_akhir']
y = clean_df['polarity']

# Verify no NaN remain
print(f"\nFinal check - NaN in X: {X.isna().sum()}")
print(f"Final check - NaN in y: {y.isna().sum()}")
print(f"Class distribution:\n{y.value_counts()}")

# Split data
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TF-IDF Vectorization
tfidf = TfidfVectorizer(
    max_features=5000,  # Increase from 5000 to capture more vocabulary
    ngram_range=(1, 3)  # Add trigrams (sequences of 3 words)
)

X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

print(f"\n✅ Success! X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

Initial data shape: (2866, 19)
NaN values in 'text_akhir': 0
NaN values in 'polarity': 0
Shape after removing NaN: (2866, 19)
Shape after removing empty strings: (2866, 19)

Final check - NaN in X: 0
Final check - NaN in y: 0
Class distribution:
polarity
negative    2260
positive     487
neutral      119
Name: count, dtype: int64

✅ Success! X_train shape: (2292, 5000)
X_test shape: (574, 5000)


In [20]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

linear = LinearSVC(C=1.0, random_state=42)
linear.fit(X_train, y_train)

y_pred_train_linear = linear.predict(X_train)
y_pred_test_linear = linear.predict(X_test)

print("=== TRAIN SET METRICS ===")
print(classification_report(y_train, y_pred_train_linear))

print("=== TEST SET METRICS ===")
print(classification_report(y_test, y_pred_test_linear))

print("=== CONFUSION MATRIX (Test) ===")
print(confusion_matrix(y_test, y_pred_test_linear))

=== TRAIN SET METRICS ===
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00      1807
     neutral       1.00      0.96      0.98        95
    positive       1.00      0.99      1.00       390

    accuracy                           1.00      2292
   macro avg       1.00      0.98      0.99      2292
weighted avg       1.00      1.00      1.00      2292

=== TEST SET METRICS ===
              precision    recall  f1-score   support

    negative       0.88      0.96      0.92       453
     neutral       0.20      0.04      0.07        24
    positive       0.70      0.54      0.61        97

    accuracy                           0.85       574
   macro avg       0.59      0.51      0.53       574
weighted avg       0.82      0.85      0.83       574

=== CONFUSION MATRIX (Test) ===
[[436   0  17]
 [ 18   1   5]
 [ 41   4  52]]


In [21]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

naive_bayes = MultinomialNB()

naive_bayes.fit(X_train, y_train)

y_pred_train_nb = naive_bayes.predict(X_train)
y_pred_test_nb = naive_bayes.predict(X_test)

accuracy_train_nb = accuracy_score(y_train, y_pred_train_nb)
accuracy_test_nb = accuracy_score(y_test, y_pred_test_nb)

print("=== TRAIN SET METRICS ===")
print(classification_report(y_train, y_pred_train_nb))

print("=== TEST SET METRICS ===")
print(classification_report(y_test, y_pred_test_nb))

print("=== CONFUSION MATRIX (Test) ===")
print(confusion_matrix(y_test, y_pred_test_nb))

=== TRAIN SET METRICS ===
              precision    recall  f1-score   support

    negative       0.79      1.00      0.88      1807
     neutral       0.00      0.00      0.00        95
    positive       1.00      0.03      0.06       390

    accuracy                           0.79      2292
   macro avg       0.60      0.34      0.31      2292
weighted avg       0.79      0.79      0.71      2292

=== TEST SET METRICS ===
              precision    recall  f1-score   support

    negative       0.79      1.00      0.88       453
     neutral       0.00      0.00      0.00        24
    positive       1.00      0.01      0.02        97

    accuracy                           0.79       574
   macro avg       0.60      0.34      0.30       574
weighted avg       0.79      0.79      0.70       574

=== CONFUSION MATRIX (Test) ===
[[453   0   0]
 [ 24   0   0]
 [ 96   0   1]]


c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

random_forest = RandomForestClassifier()

random_forest.fit(X_train.toarray(), y_train)

y_pred_train_rf = random_forest.predict(X_train.toarray())
y_pred_test_rf = random_forest.predict(X_test.toarray())

accuracy_train_rf = accuracy_score(y_pred_train_rf, y_train)
accuracy_test_rf = accuracy_score(y_pred_test_rf, y_test)

print("=== TRAIN SET METRICS ===")
print(classification_report(y_train, y_pred_train_rf))

print("=== TEST SET METRICS ===")
print(classification_report(y_test, y_pred_test_rf))

print("=== CONFUSION MATRIX (Test) ===")
print(confusion_matrix(y_test, y_pred_test_rf))

=== TRAIN SET METRICS ===
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00      1807
     neutral       1.00      1.00      1.00        95
    positive       1.00      1.00      1.00       390

    accuracy                           1.00      2292
   macro avg       1.00      1.00      1.00      2292
weighted avg       1.00      1.00      1.00      2292

=== TEST SET METRICS ===
              precision    recall  f1-score   support

    negative       0.81      0.99      0.89       453
     neutral       0.00      0.00      0.00        24
    positive       0.74      0.14      0.24        97

    accuracy                           0.81       574
   macro avg       0.52      0.38      0.38       574
weighted avg       0.77      0.81      0.75       574

=== CONFUSION MATRIX (Test) ===
[[450   0   3]
 [ 22   0   2]
 [ 82   1  14]]


In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

logistic_regression = LogisticRegression()

logistic_regression.fit(X_train.toarray(), y_train)

y_pred_train_lr = logistic_regression.predict(X_train.toarray())
y_pred_test_lr = logistic_regression.predict(X_test.toarray())

accuracy_train_lr = accuracy_score(y_pred_train_lr, y_train)

accuracy_test_lr = accuracy_score(y_pred_test_lr, y_test)

print("=== TRAIN SET METRICS ===")
print(classification_report(y_train, y_pred_train_lr))

print("=== TEST SET METRICS ===")
print(classification_report(y_test, y_pred_test_lr))

print("=== CONFUSION MATRIX (Test) ===")
print(confusion_matrix(y_test, y_pred_test_lr))

=== TRAIN SET METRICS ===
              precision    recall  f1-score   support

    negative       0.86      1.00      0.93      1807
     neutral       0.00      0.00      0.00        95
    positive       0.99      0.52      0.68       390

    accuracy                           0.88      2292
   macro avg       0.62      0.51      0.54      2292
weighted avg       0.85      0.88      0.85      2292

=== TEST SET METRICS ===
              precision    recall  f1-score   support

    negative       0.83      0.99      0.90       453
     neutral       0.00      0.00      0.00        24
    positive       0.88      0.29      0.43        97

    accuracy                           0.83       574
   macro avg       0.57      0.43      0.45       574
weighted avg       0.80      0.83      0.79       574

=== CONFUSION MATRIX (Test) ===
[[449   0   4]
 [ 24   0   0]
 [ 69   0  28]]


c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [24]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

decision_tree = DecisionTreeClassifier()

decision_tree.fit(X_train.toarray(), y_train)

y_pred_train_dt = decision_tree.predict(X_train.toarray())
y_pred_test_dt = decision_tree.predict(X_test.toarray())

accuracy_train_dt = accuracy_score(y_pred_train_dt, y_train)
accuracy_test_dt = accuracy_score(y_pred_test_dt, y_test)

print("=== TRAIN SET METRICS ===")
print(classification_report(y_train, y_pred_train_dt))

print("=== TEST SET METRICS ===")
print(classification_report(y_test, y_pred_test_dt))

print("=== CONFUSION MATRIX (Test) ===")
print(confusion_matrix(y_test, y_pred_test_dt))

=== TRAIN SET METRICS ===
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00      1807
     neutral       1.00      1.00      1.00        95
    positive       1.00      1.00      1.00       390

    accuracy                           1.00      2292
   macro avg       1.00      1.00      1.00      2292
weighted avg       1.00      1.00      1.00      2292

=== TEST SET METRICS ===
              precision    recall  f1-score   support

    negative       0.84      0.87      0.85       453
     neutral       0.12      0.12      0.12        24
    positive       0.41      0.34      0.37        97

    accuracy                           0.75       574
   macro avg       0.46      0.44      0.45       574
weighted avg       0.74      0.75      0.74       574

=== CONFUSION MATRIX (Test) ===
[[393  17  43]
 [ 16   3   5]
 [ 59   5  33]]
